In [1]:
# data_pull.py
# downloads portfolio return data from Kenneth French's data library
# via pandas-datareader and builds the X matrix for the replication
#
# we originally looked at pulling all 37 anomalies directly from WRDS/CRSP
# but constructing those from scratch is roughly 2 weeks of work. French's
# library gives us 10 of the main ones which is enough to demonstrate the
# methodology -- mean returns and spreads all check out economically.
#
# output: X_large_cross_section.csv, X_double_sort.csv, ff5_factors.csv

# %%

import pandas as pd
import numpy as np
import pandas_datareader.data as web
import warnings
warnings.filterwarnings("ignore")

START = "1963-07"
END   = "2017-12"


# %%

#___ helper to pull one dataset from French library ___

def get_french(name, start="1963-01-01", end="2018-01-01"):
    raw = web.DataReader(name, "famafrench", start=start, end=end)
    df  = raw[0].copy()
    df.index = pd.to_datetime(df.index.to_timestamp())
    df = df / 100
    return df


# %%

#___ download all portfolios we need ___

print("pulling from French library...")

port_25_bm  = get_french("25_Portfolios_5x5")            # size x B/M (double sort)
port_25_mom = get_french("25_Portfolios_ME_Prior_12_2")   # size x momentum

"""
single sorted decile portfolios for the 10 anomalies.
French bundles deciles + quintiles + thirds in one table so the shape
is usually 18-19 columns -- the actual decile portfolios are always the
last 10 columns (Lo10, 2-Dec ... Hi10). confirmed by checking col names.
"""
port_bm    = get_french("Portfolios_Formed_on_BE-ME")
port_ep    = get_french("Portfolios_Formed_on_E-P")
port_cfp   = get_french("Portfolios_Formed_on_CF-P")
port_dp    = get_french("Portfolios_Formed_on_D-P")
port_mom   = get_french("10_Portfolios_Prior_12_2")    # already 10 cols
port_strev = get_french("10_Portfolios_Prior_1_0")     # short-term reversal
port_ltrev = get_french("10_Portfolios_Prior_60_13")   # long-term reversal
port_inv   = get_french("Portfolios_Formed_on_INV")
port_prof  = get_french("Portfolios_Formed_on_OP")
port_size  = get_french("Portfolios_Formed_on_ME")

ff5 = get_french("F-F_Research_Data_5_Factors_2x3")
rf  = ff5["RF"]

print("downloads done\n")


# %%

#___ build excess return matrix X ___

"""
each anomaly contributes 10 decile portfolios in excess of RF,
giving us N = 100 portfolios total across T months
"""

all_ports = {
    "bm":    port_bm,
    "ep":    port_ep,
    "cfp":   port_cfp,
    "dp":    port_dp,
    "mom":   port_mom,
    "strev": port_strev,
    "ltrev": port_ltrev,
    "inv":   port_inv,
    "prof":  port_prof,
    "size":  port_size,
}

print(f"{'anomaly':10s}  {'total cols':>10s}  {'last 10 (deciles)':>40s}")
print("-" * 65)

frames = []
for label, df in all_ports.items():
    d10 = df.iloc[:, -10:].copy()
    d10.columns = [f"{label}_d{i+1}" for i in range(10)]
    d10 = d10.subtract(rf, axis=0)
    frames.append(d10)
    print(f"{label:10s}  {len(df.columns):>10d}  {list(df.columns[-10:])}")

X_ret = pd.concat(frames, axis=1).dropna()
X_ret = X_ret.loc[START:END]

print(f"\nX shape:    {X_ret.shape}")
print(f"date range: {X_ret.index[0].strftime('%Y-%m')} to {X_ret.index[-1].strftime('%Y-%m')}")


# %%

# double sort matrix (25 size x BM portfolios) for section 4.1 equivalent

X_ds = port_25_bm.iloc[:, :25].copy()
X_ds = X_ds.subtract(rf, axis=0).dropna()
X_ds = X_ds.loc[START:END]
print(f"double sort shape: {X_ds.shape}")


# %%

#___ sanity check: mean excess returns and spreads ___

"""
expected signs:
  bm, ep, cfp, dp  -- positive spread (value premium)
  mom              -- large positive spread
  strev, ltrev     -- negative spread (reversal)
  inv              -- negative spread (high investment -> lower returns)
  size             -- negative spread (small in D1 outperforms large in D10)
"""

print(f"\n{'anomaly':10s}  {'mean%':>7s}  {'D1 (low)':>10s}  {'D10 (high)':>10s}  {'spread':>8s}")
print("-" * 55)
for label in all_ports:
    cols = [c for c in X_ret.columns if c.startswith(label)]
    avg  = X_ret[cols].mean().mean() * 100
    d1   = X_ret[cols[0]].mean()     * 100
    d10  = X_ret[cols[-1]].mean()    * 100
    sprd = d10 - d1
    print(f"{label:10s}  {avg:>7.3f}  {d1:>10.3f}  {d10:>10.3f}  {sprd:>8.3f}")

print(f"\nany NaN:                {X_ret.isnull().any().any()}")
print(f"mean monthly excess ret: {X_ret.values.mean()*100:.3f}%  (expecting ~0.4 to 0.6)")


# %%

#___ save ___

X_ret.to_csv("X_large_cross_section.csv")
X_ds.to_csv("X_double_sort.csv")
ff5.loc[START:END].to_csv("ff5_factors.csv")
print("saved X_large_cross_section.csv, X_double_sort.csv, ff5_factors.csv")
print("next: replicate.py")

pulling from French library...
downloads done

anomaly     total cols                         last 10 (deciles)
-----------------------------------------------------------------
bm                  19  ['Lo 10', '2-Dec', '3-Dec', '4-Dec', '5-Dec', '6-Dec', '7-Dec', '8-Dec', '9-Dec', 'Hi 10']
ep                  19  ['Lo 10', '2-Dec', '3-Dec', '4-Dec', '5-Dec', '6-Dec', '7-Dec', '8-Dec', '9-Dec', 'Hi 10']
cfp                 19  ['Lo 10', '2-Dec', '3-Dec', '4-Dec', '5-Dec', '6-Dec', '7-Dec', '8-Dec', '9-Dec', 'Hi 10']
dp                  19  ['Lo 10', '2-Dec', '3-Dec', '4-Dec', '5-Dec', '6-Dec', '7-Dec', '8-Dec', '9-Dec', 'Hi 10']
mom                 10  ['Lo PRIOR', 'PRIOR 2', 'PRIOR 3', 'PRIOR 4', 'PRIOR 5', 'PRIOR 6', 'PRIOR 7', 'PRIOR 8', 'PRIOR 9', 'Hi PRIOR']
strev               10  ['Lo PRIOR', 'PRIOR 2', 'PRIOR 3', 'PRIOR 4', 'PRIOR 5', 'PRIOR 6', 'PRIOR 7', 'PRIOR 8', 'PRIOR 9', 'Hi PRIOR']
ltrev               10  ['Lo PRIOR', 'PRIOR 2', 'PRIOR 3', 'PRIOR 4', 'PRIOR 5', 'PRIOR 